# Real data: the Oxford mouse benchmark

Everything so far ran on synthetic signal, which is built to contain exactly the
structure nyx looks for and therefore says nothing about how well it works. This
notebook uses a real recording.

**One download, 78 MB.** The Oxford Mouse Polysomnography Benchmark's *pilot*
set is a 12 h EEG/EMG recording from a freely behaving mouse during the light
phase, scored independently by ten experienced sleep researchers. We compare
against their **consensus**, which is the reference the dataset provides for
exactly this purpose.

> Cite the dataset if you use it:
> Brodersen et al., *The Oxford Mouse Polysomnography Benchmark Data Set*,
> Zenodo, [10.5281/zenodo.10200482](https://doi.org/10.5281/zenodo.10200482).
> CC BY 4.0.

## 1. Setup

In [ ]:
import os

for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = "1"

import glob
import io
import urllib.request
import zipfile

import matplotlib.pyplot as plt

import nyx

## 2. Download it

78 MB, into `examples/data/` (which is gitignored). Skipped if already there, so
this cell is safe to re-run.

The other three groups in the record are much larger — `test.zip` is 1.3 GB,
`sleep_deprivation.zip` 5.1 GB, `optogenetic_stimulation.zip` 2.8 GB. Change
`WHICH` if you want one of those, but start here.

In [ ]:
WHICH = "pilot"
RECORD = "10200482"
DATA = "data"

url = f"https://zenodo.org/api/records/{RECORD}/files/{WHICH}.zip/content"
target = os.path.join(DATA, WHICH)

if os.path.isdir(target):
    print(f"already downloaded: {target}")
else:
    os.makedirs(DATA, exist_ok=True)
    print(f"downloading {WHICH}.zip ...")
    with urllib.request.urlopen(url, timeout=600) as response:
        payload = response.read()
    print(f"  {len(payload) / 1e6:.0f} MB, extracting")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall(DATA)
    print("done")

for path in sorted(glob.glob(f"{target}/**/*", recursive=True))[:20]:
    if os.path.isfile(path):
        print(f"  {os.path.getsize(path) / 1e6:8.2f} MB  {path}")

## 3. Load it

**The channel indices are not guessable and differ between the four groups.**
For `pilot` the dataset README gives:

| | channel |
|---|---|
| frontal EEG | `Signal 0` |
| parietal EEG | `Signal 1` |
| **EMG** | **`Signal 3`** |

Note the EMG is `Signal 3`, not `Signal 1` — using position defaults of 0 and 1
would silently feed the parietal EEG in as the EMG, and everything downstream
would look plausible and be wrong. Select by name.

In [ ]:
recording = nyx.read_recording(
    f"{target}/recordings/{WHICH}.edf",
    eeg_channel="Signal 0",     # frontal EEG
    emg_channel="Signal 3",     # EMG -- not Signal 1, which is parietal EEG
    name=WHICH,
)
print(recording.describe())

In [ ]:
consensus = nyx.read_annotations(f"{target}/annotations/{WHICH}_consensus.hyp")

totals = {}
for duration, label in zip(consensus["duration"], consensus["label"]):
    totals[str(label)] = totals.get(str(label), 0.0) + float(duration)

print("consensus of 10 scorers:")
for label, seconds in sorted(totals.items()):
    print(f"  {label:<12} {seconds / 3600:5.2f} h  ({100 * seconds / 43200:.1f}%)")

The vocabulary here is `awake` / `non-REM` / `REM` / `undefined`. nyx normalises
common spellings when comparing, so `awake` matches `WAKE` and `non-REM` matches
`NREM` without you doing anything.

## 4. Look at the signal before scoring it

Always. It takes a minute and it is where you find the problems that would
otherwise turn into a plausible-looking wrong answer.

In [ ]:
check = nyx.check_signals(recording)
print(check.summary())
check.plot()
plt.show()

## 5. 🔧 Your turn

This is the part that is **yours**, not the package's. Defaults are a starting
point; a real recording usually wants a few minutes of attention. Work through
them in order, re-running the cell below after each change:

**The EMG threshold.** Look at the histogram. Does the cut sit in a valley
between two modes? If the EMG does not separate cleanly, set the threshold above
everything so all epochs start as SLEEP, and let clustering recover wake from the
EEG instead — that is what `params/mouse_weak_emg.json` does.

**The analysis window.** Trim anything the signal check showed as unusable.

**A notch**, if the check suggested one — into the params, not here, so it is
recorded and replays.

**The clustering settings and the stage order.** Check the per-cluster spectra:
NREM must carry more low-frequency power than REM. If it does not, the
assignment is wrong regardless of how clean the clusters look.

`01_score_recording.ipynb` walks through each of these one cell at a time, with
the plots that inform them. Use it if a change here does not do what you expect.

In [ ]:
params = nyx.load_params("../params/mouse.json")

# 🔧 Adjust these.
WINDOW = None            # (start, end) in seconds, or None for the whole recording
EMG_THRESHOLD = None     # a number to override the automatic cut
STAGE_ORDER = ("REM", "NREM")
CLUSTER_OVERRIDES = None # e.g. {0: "NREM", 1: "REM"}

# params["EEG"]["notch"] = 50
# params["EMG"]["notch"] = 50
# params["clustering"] = {"method": "gmm", "n_clusters": 2, "pcs_to_use": [0, 1, 2]}

result = nyx.score_recording(
    recording, params,
    window=WINDOW,
    emg_threshold=EMG_THRESHOLD,
    stage_order=STAGE_ORDER,
    cluster_overrides=CLUSTER_OVERRIDES,
    reference=consensus,
    verbose=True,
)

In [ ]:
nyx.plot_summary(result)
plt.show()

In [ ]:
nyx.plot_scoring_overview(result)
plt.show()

## 6. Save

In [ ]:
OUTPUT_DIR = f"../results/{WHICH}"
nyx.save_results(result, OUTPUT_DIR)
print("saved to", os.path.abspath(OUTPUT_DIR))

`run.json` records the parameters and every decision you made above, so this
scoring replays exactly — including the ones you had to think about.

---

## The other three groups

Same code, different `WHICH` and different channels. Check the dataset README
for each: the channel indices are **not** the same between groups.

| group | size | contents |
|---|---|---|
| `pilot` | 78 MB | 1 recording, 12 h, 10 scorers |
| `test` | 1.3 GB | 6 recordings, 24 h, ≥4 scorers each |
| `sleep_deprivation` | 5.1 GB | sleep deprivation protocol |
| `optogenetic_stimulation` | 2.8 GB | optogenetic stimulation |

To score several of them, put them in a manifest and use the queue —
`03_batch_queue.ipynb`.